### Low rank analysis - Time evolution
Import libraries

In [1]:
import os
import sys

libs = {
  "./../modules",
  "/u/zanardi3/Codes/ML/PyCOMET/pycomet"
}
for lib in libs:
  sys.path.append(lib)

In [2]:
import json
import numpy as np
import scipy as sp
import dask.array as da

from silx.io.dictdump import dicttoh5, h5todict

In [3]:
from converter import Converter

Define inputs

In [19]:
# Path to database
path_to_dtb = "./../../database/"
# Path to 0D solutions
path_to_sols = [
  "./../../datagen/runs/samples/data/train/sols.hdf5",
  "./../../datagen/runs/samples/data/valid/sols.hdf5"
]
# Time domain
t_lim = [1e-9, 1e-5]
# Variables to decompose
variables = {
  "n_em": r"$\tilde{n}_{e^-}$",
  "n_Ar1": r"$\tilde{n}_{Ar_1}$",
  "pe": r"$\tilde{p}_e$",
  "Te": r"$\tilde{T}_e$",
  "Th": r"$\tilde{T}_h$"
}
# Path to saving
path_to_save = "./../output/lowrank_time/eps3/"
os.makedirs(path_to_save, exist_ok=True)

Read solutions

In [5]:
conv = Converter(path_to_dtb)

In [6]:
sols = []
for p in path_to_sols:
  sols.extend(list(h5todict(p).values()))
phys = [conv.prim_to_phys(**sol) for sol in sols]

Perfoming SVD
> Interpolate solutions onto the same grid

In [7]:
def interpolate(t, var, phys):
  x = phys["t"].squeeze()
  y = phys[var].squeeze()
  # Extend domain
  x = np.concatenate([x, x[-1:]*1e2])
  y = np.concatenate([y, y[-1:]])
  # Interpolate
  return sp.interpolate.interp1d(x, y, kind='cubic')(t)

In [8]:
y = {}
t = np.geomspace(*t_lim, 200)
for k in variables.keys():
  yk = [interpolate(t=t, var=k, phys=p) for p in phys]
  y[k] = np.vstack(yk)

> SVD decomposition

In [9]:
svd = {}
for k in variables.keys():
  # Standardize
  yk = np.log(y[k])
  mean = np.mean(yk, axis=0)
  std = np.std(yk, axis=0)
  yk = (yk - mean) / std
  # Perform SVD
  yk = da.from_array(yk.T, chunks="auto")
  u, s, vh = da.linalg.svd_compressed(yk, k=100)
  s = s.compute()
  u = u.compute()
  # Storing
  svd[k] = {
    "mean": mean,
    "std": std,
    "s": s,
    "u": u,
  }

> Singular values decay

In [20]:
eps = 1e-6
phi, rdim = {}, {"eps": eps}
for k in variables.keys():
  # Cumulative energy
  s = svd[k]["s"]
  cs = 1.0 / np.sum(s**2)
  cs *= np.cumsum(s**2)
  # Number of principal components
  dim = np.where(cs > 1-eps)[0][0] + 1
  rdim[k] = int(dim)
  # Extract modes
  svd[k]["phi"] = svd[k]["u"][:,:dim]
# Save number of principal components
with open(path_to_save + "/rdim.json", "w") as file:
  json.dump(rdim, file, indent=2, ensure_ascii=False)

> Test reconstruction

In [21]:
def reconstruct(y, svd):
  phi, m, s = [svd[k] for k in ("phi", "mean", "std")]
  y = np.log(y)
  y = (y - m) / s
  a = y @ phi
  y = (a @ phi.T) * s + m
  y = np.exp(y)
  return y

def error(yt, yp):
  return 100*np.abs(yp-yt)/(np.abs(yt)+1e-15)

In [22]:
err = {}
for (k, label) in variables.items():
  yr = reconstruct(y[k], svd[k])
  err[k] = error(y[k], yr).max()
# Save number of principal components
with open(path_to_save + "/err_max.json", "w") as file:
  json.dump(err, file, indent=2, ensure_ascii=False)

> Save tensors

In [23]:
for k in variables.keys():
  phi = svd[k]["phi"].T * svd[k]["std"]
  phi = np.vstack([phi, svd[k]["mean"]]).T
  dicttoh5(
    treedict={"x": t.reshape(-1,1), "y": phi},
    h5file=path_to_save + f"/basis_{k}.hdf5"
  )